# L7: Build a Crew to Tailor Job Applications

In this lesson, you will built your first multi-agent system.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import load_env
load_env()

## crewAI Tools

In [ ]:
from crewai_tools import FileReadTool

from utils import BaiduWebSearchTool, ScrapeWebsiteTool, create_resume_search_tool

search_tool = BaiduWebSearchTool()
scrape_tool = ScrapeWebsiteTool()

read_resume = FileReadTool(file_path='./fake_resume.md')
# semantic_search_resume = create_resume_search_tool('./fake_resume.md')

- Uncomment and run the cell below if you wish to view `fake_resume.md` in the notebook.

In [ ]:
# from IPython.display import Markdown, display
# display(Markdown("./fake_resume.md"))

## Creating Agents

In [ ]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="对职位发布进行出色分析，以帮助求职者",
    tools = [scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "作为职位研究员，你擅长浏览职位发布并提取关键信息，"
        "能力无人能及。你能精准识别雇主所需的资格与技能，"
        "为高效定制求职申请奠定基础。"
    )
)

In [ ]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="对求职者进行深入研究，帮助他们在就业市场中脱颖而出",
    tools = [scrape_tool, search_tool,
             read_resume],
    verbose=True,
    backstory=(
        "你具备出色的分析能力，能从多种来源拆解并综合信息，"
        "撰写全面的个人与职业画像，为个性化简历优化打下基础。"
    )
)

In [ ]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="找到所有最佳方法，让简历在就业市场中脱颖而出",
    tools = [scrape_tool, search_tool,
             read_resume],
    verbose=True,
    backstory=(
        "你具备战略思维与细致眼光，擅长优化简历以突出"
        "最相关的技能与经历，确保与职位要求高度契合。"
    )
)

In [ ]:
# Agent 4: Interview Preparer
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="根据简历与职位要求，制定面试问题与谈话要点",
    tools = [scrape_tool, search_tool,
             read_resume],
    verbose=True,
    backstory=(
        "你的角色对预判面试动态至关重要。你能拟定关键问题"
        "与谈话要点，帮助候选人充分准备，自信应对所申请职位"
        "的各个方面。"
    )
)

## Creating Tasks

In [ ]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "分析提供的职位发布链接（{job_posting_url}），"
        "提取所需的关键技能、经历与资格要求。"
        "使用工具收集内容，识别并对要求进行分类。"
    ),
    expected_output=(
        "结构化的职位要求清单，包括必要的技能、资格与经历。"
    ),
    agent=researcher,
    async_execution=True
)

In [ ]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "根据 GitHub 链接（{github_url}）与个人简介"
        "（{personal_writeup}），整理详细的个人与职业画像。"
        "使用工具从这些来源提取并综合信息。"
    ),
    expected_output=(
        "全面的画像文档，涵盖技能、项目经历、贡献、"
        "兴趣与沟通风格。"
    ),
    agent=profiler,
    async_execution=True
)

- You can pass a list of tasks as `context` to a task.
- The task then takes into account the output of those tasks in its execution.
- The task will not run until it has the output(s) from those tasks.

In [ ]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "根据前序任务获得的画像与职位要求，定制简历以突出"
        "最相关的内容。使用工具调整并优化简历内容。"
        "务必打造最佳简历，但不得编造任何信息。"
        "更新所有章节，包括开篇摘要、工作经历、技能与教育背景，"
        "以更好地体现候选人能力及其与职位发布的匹配度。"
    ),
    expected_output=(
        "更新后的简历，有效突出与职位相关的候选人资格与经历。"
    ),
    output_file="tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [ ]:
# Task for Interview Preparer Agent: Develop Interview Materials
interview_preparation_task = Task(
    description=(
        "根据定制简历与职位要求，制定一套潜在的面试问题与谈话要点。"
        "使用工具生成相关问题与讨论要点。"
        "确保利用这些问题与谈话要点，帮助候选人突出简历要点"
        "及其与职位发布的匹配之处。"
    ),
    expected_output=(
        "包含候选人应在初试中准备的关键问题与谈话要点的文档。"
    ),
    output_file="interview_materials.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)


## Creating the Crew

In [ ]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [ ]:
job_application_inputs = {
    'job_posting_url': 'https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund',
    'github_url': 'https://github.com/joaomdmoura',
    'personal_writeup': """Noah 是一位经验丰富的软件工程领导者，拥有 18 年从业经历，
    擅长管理远程与现场团队，精通多种编程语言与框架。
    他拥有 MBA 学位，在人工智能与数据科学领域背景扎实。
    Noah 曾成功主导多项重大技术项目与创业公司，
    展现出推动科技行业创新与增长的能力。
    非常适合需要战略思维与创新方法的领导岗位。"""
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=job_application_inputs)

- Dislplay the generated `tailored_resume.md` file.

In [ ]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

- Dislplay the generated `interview_materials.md` file.

In [ ]:
display(Markdown("./interview_materials.md"))

# CONGRATULATIONS!!!

## Share your accomplishment!
- Once you finish watching all the videos, you will see the "In progress" image on the bottom left turn into "Accomplished".
- Click on "Accomplished" to view the course completion page with your name on it.
- Take a screenshot and share on LinkedIn, X (Twitter), or Facebook.  
- **Tag @Joāo (Joe) Moura, @crewAI, and @DeepLearning.AI, (and a few of your friends if you'd like them to try out the course)**
- **Joāo and DeepLearning.AI will "like"/reshare/comment on your post!**

## Get a completion badge that you can add to your LinkedIn profile!
- Go to [learn.crewai.com](https://learn.crewai.com).
- Upload your screenshot of your course completion page.
- You'll get a badge from CrewAI that you can share!

(Joāo will also talk about this in the last video of the course.)